###Ingest customers file

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("city", StringType(), True),
    StructField("signup_date", DateType(), True)
])

In [0]:
customers_raw_df = (
    spark.read
        .option("header", "true")
        .schema(customers_schema)
        .csv("abfss://raw@customersegmentationsa.dfs.core.windows.net/customers.csv")
)

In [0]:
customers_raw_df.show()

+-----------+------+----+------+---------+-----------+
|customer_id|  name| age|gender|     city|signup_date|
+-----------+------+----+------+---------+-----------+
|       C001| Rahul|  29|     M|Bangalore| 2023-01-10|
|       C002| Anita|  35|     F|   Mumbai| 2022-11-05|
|       C003|Suresh|  42|     M|    Delhi| 2021-06-20|
|       C004| Priya|  27|     F|  Chennai| 2023-03-15|
|       C005|  Amit|  38|     M|     Pune| 2020-08-12|
|       C006|  Neha|NULL|     F|Bangalore| 2023-05-09|
|       C007|  Ravi|  31|     M|     NULL| 2022-02-18|
|       C008|Kavita|  45|     F|  Kolkata| 2019-12-01|
|       C009| Manoj|  52|     M|    Delhi| 2018-07-25|
|       C010|Sunita|  33|     F|   Mumbai| 2023-06-30|
+-----------+------+----+------+---------+-----------+



###Apply data quality & standardisation rules

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    initcap,
    upper,
    when,
    current_timestamp
)

customers_processed_df = (
    customers_raw_df
        # Remove records without customer_id
        .filter(col("customer_id").isNotNull())

        # Standardize text fields
        .withColumn("customer_id", trim(col("customer_id")))
        .withColumn("name", initcap(trim(col("name"))))
        .withColumn("city", initcap(trim(col("city"))))
        .withColumn("gender", upper(trim(col("gender"))))

        # Handle invalid ages
        .withColumn(
            "age",
            when((col("age") < 0) | (col("age") > 120), None)
            .otherwise(col("age"))
        )

        # Add derived columns
        .withColumn(
            "age_group",
            when(col("age").between(18, 25), "18-25")
            .when(col("age").between(26, 35), "26-35")
            .when(col("age").between(36, 45), "36-45")
            .when(col("age") > 45, "46+")
            .otherwise("Unknown")
        )

        # Audit column
        .withColumn("processed_timestamp", current_timestamp())
)

In [0]:
display(customers_processed_df)

customer_id,name,age,gender,city,signup_date,age_group,processed_timestamp
C001,Rahul,29,M,Bangalore,2023-01-10,26-35,2026-03-27T12:15:47.754633Z
C002,Anita,35,F,Mumbai,2022-11-05,26-35,2026-03-27T12:15:47.754633Z
C003,Suresh,42,M,Delhi,2021-06-20,36-45,2026-03-27T12:15:47.754633Z
C004,Priya,27,F,Chennai,2023-03-15,26-35,2026-03-27T12:15:47.754633Z
C005,Amit,38,M,Pune,2020-08-12,36-45,2026-03-27T12:15:47.754633Z
C006,Neha,null,F,Bangalore,2023-05-09,Unknown,2026-03-27T12:15:47.754633Z
C007,Ravi,31,M,null,2022-02-18,26-35,2026-03-27T12:15:47.754633Z
C008,Kavita,45,F,Kolkata,2019-12-01,36-45,2026-03-27T12:15:47.754633Z
C009,Manoj,52,M,Delhi,2018-07-25,46+,2026-03-27T12:15:47.754633Z
C010,Sunita,33,F,Mumbai,2023-06-30,26-35,2026-03-27T12:15:47.754633Z


###Save the output as delta table to processed container

In [0]:
customers_processed_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("processed_db.customers")

In [0]:
%sql
select * from processed_db.customers;

customer_id,name,age,gender,city,signup_date,age_group,processed_timestamp
C001,Rahul,29,M,Bangalore,2023-01-10,26-35,2026-03-27T12:15:48.988293Z
C002,Anita,35,F,Mumbai,2022-11-05,26-35,2026-03-27T12:15:48.988293Z
C003,Suresh,42,M,Delhi,2021-06-20,36-45,2026-03-27T12:15:48.988293Z
C004,Priya,27,F,Chennai,2023-03-15,26-35,2026-03-27T12:15:48.988293Z
C005,Amit,38,M,Pune,2020-08-12,36-45,2026-03-27T12:15:48.988293Z
C006,Neha,null,F,Bangalore,2023-05-09,Unknown,2026-03-27T12:15:48.988293Z
C007,Ravi,31,M,null,2022-02-18,26-35,2026-03-27T12:15:48.988293Z
C008,Kavita,45,F,Kolkata,2019-12-01,36-45,2026-03-27T12:15:48.988293Z
C009,Manoj,52,M,Delhi,2018-07-25,46+,2026-03-27T12:15:48.988293Z
C010,Sunita,33,F,Mumbai,2023-06-30,26-35,2026-03-27T12:15:48.988293Z
